# Notebook 04 — Discretização das Variáveis

**Projeto:** Mineração de Dados em Saúde · PNS 2019
**Estudo:** Artrite e Reumatismo em Idosos Brasileiros
**Pesquisador:** Pedro Dias Soares
**Orientador:** Prof. Dr. Luis Enrique Zárate — PUC Minas (LICAP)

---

## Por que esta etapa existe (e por que num notebook próprio)

A discretização é a **etapa 7 do KDD** (slide *Introdução12 — Preparação dos dados*) e foi
isolada num notebook dedicado para deixar **cada corte auditável e rastreável**. O slide a
define como "transformar atributos quantitativos em qualitativos com número finito de
intervalos", com três ganhos: (1) algoritmos como Árvore de Decisão/Naïve Bayes operam melhor
sobre categóricos; (2) **redução de dimensionalidade**; (3) **suavização de outliers** (valores
extremos caem dentro de um nível) — argumento explícito no artigo de *Hipertensão (CBIS'24)*.

## Como os artigos do orientador discretizam (padrão seguido aqui)

Verifiquei 6 artigos do LICAP sobre a PNS (AVC, Depressão, Artrite+Depressão, TOC, DPOC,
Hipertensão). O padrão é **uniforme**:

- Discretização **não-supervisionada, univariada, por faixas de domínio** — cortes vindos de
  fontes oficiais (OMS, NIAAA, IBGE, Ministério da Saúde, Guia de Atividade Física BR 2021),
  **não** dos dados.
- "Risco crescente = nível maior" (categórico **ordinal**), para favorecer interpretabilidade.
- **Entropia + correlação** aparecem, mas para **selecionar/remover atributos** — *nunca* para
  definir os cortes. Este notebook respeita essa divisão.

## Pipeline (Opção A — fonte única de discretização)

Os notebooks **03** (artrite pura) e **03b** (artrite + comorbidades) entregam as variáveis
contínuas ainda **numéricas** (a antiga "Etapa 10" de categorização foi migrada para cá). Assim
não há discretização duplicada — a mesma lógica que isolou o tratamento de outliers.

## Este notebook processa OS DOIS desenhos

- **Parte A (passo a passo):** roda um desenho à escolha (`DESENHO`), célula a célula, com os
  gráficos de diagnóstico — para você inspecionar e calibrar os cortes.
- **Parte B (lote):** a Seção 12 reúne a lógica numa função e executa **ambos** os desenhos de
  uma vez, gravando os dois `dataset_discretizado.csv` + os dois `relatorio_discretizacao.json`.

| # | Etapa | O que faz |
|---|-------|-----------|
| 1 | Configuração | Imports, caminhos por desenho, paleta, helpers |
| 2 | Carregamento | Lê o dataset pré-processado do desenho ativo |
| 3 | Mapa de discretização | Faixas **+ fonte** de cada corte (domínio e quantis) |
| 4 | Inventário | O que ainda é contínuo × o que já é categórico |
| 5 | Diagnóstico ANTES | Histogramas por intervalo (slide) |
| 6 | Discretização por domínio | Binning não-supervisionado com faixas oficiais |
| 7 | (Opcional) equal-frequency | `qcut` só p/ escores sem referência externa |
| 8 | Perda de informação | Prevalência da classe por faixa (Figuras dos artigos) |
| 9 | Seleção entropia+correlação | Poda/sinalização de atributos (padrão dos artigos) |
| 10 | Codificação ordinal | `_cat` como inteiro 0..k e remoção das contínuas originais |
| 11 | Exportação | CSV + JSON do desenho ativo |
| 12 | **Lote** | Roda os DOIS desenhos de uma vez |

> Convenção de cor: <span style="color:#C0392B">**vermelho = Com Artrite**</span>,
> <span style="color:#27AE60">**verde = Saudável**</span>.

## 1 · Configuração do ambiente

Mesma base do NB03 (paleta, helpers, `RANDOM_STATE=42`). O dicionário `CONFIG_DESENHOS`
centraliza, para cada desenho, **de onde ler** (saída do 03/03b) e **onde gravar**. A variável
`DESENHO` escolhe qual desenho a *Parte A* percorre passo a passo. A *Parte B* (Seção 12) usa
todos os desenhos do dicionário.

In [ ]:
import os, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Mapa dos dois desenhos: entrada (saída do NB03/03b) e saída (deste NB) ──
CONFIG_DESENHOS = {
    'puro': {
        'rotulo':   'Desenho 1 — Artrite pura',
        'entrada':  '../data/results/preprocessing/dataset_preprocessado.csv',
        'saida':    '../data/results/discretizacao/',
    },
    'comorbidades': {
        'rotulo':   'Desenho 2 — Artrite + comorbidades',
        'entrada':  '../data/results/preprocessing_comorbidades/dataset_preprocessado.csv',
        'saida':    '../data/results/discretizacao_comorbidades/',
    },
}

# Desenho percorrido na PARTE A (passo a passo). A PARTE B roda os dois.
DESENHO = 'puro'

RANDOM_STATE = 42
COR_ARTRITE  = '#C0392B'   # vermelho (Com Artrite)
COR_SAUDAVEL = '#27AE60'   # verde    (Saudável)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

def preparar_dirs(saida):
    figs, tabs = saida + 'figuras/', saida + 'tabelas/'
    for p in (saida, figs, tabs):
        os.makedirs(p, exist_ok=True)
    return figs, tabs

def salvar_fig(figs, nome):
    plt.savefig(figs + nome, dpi=150, bbox_inches='tight'); print(f'  ✅ Figura → {figs+nome}')

def salvar_tab(tabs, df, nome):
    df.to_csv(tabs + nome, encoding='utf-8-sig');           print(f'  ✅ Tabela → {tabs+nome}')

# Dirs do desenho ATIVO (Parte A)
cfg = CONFIG_DESENHOS[DESENHO]
DIR_RESULTADOS = cfg['saida']
DIR_FIGURAS, DIR_TABELAS = preparar_dirs(DIR_RESULTADOS)

print(f'✅ Ambiente configurado. Desenho ativo (Parte A) = {DESENHO!r} — {cfg["rotulo"]}')
print(f'   Entrada : {cfg["entrada"]}')
print(f'   Saída   : {DIR_RESULTADOS}')

## 2 · Carregamento do dataset pré-processado

Lê o `dataset_preprocessado.csv` do desenho ativo. A coluna `Label` (0=Saudável, 1=Com Artrite)
é separada e **nunca** é discretizada — só é usada nas Etapas 8/9 (perda de informação e seleção
supervisionada). O `assert` protege contra rodar antes do NB03 correspondente.

In [ ]:
caminho_in = cfg['entrada']
assert os.path.exists(caminho_in), f'Arquivo não encontrado: {caminho_in}. Rode o NB03/03b antes.'
df = pd.read_csv(caminho_in)
print(f'  ✅ Carregado: {df.shape[0]:,} registros × {df.shape[1]} colunas')

assert 'Label' in df.columns, "Coluna 'Label' não encontrada — confira a saída do NB03."
y = df['Label'].astype(int)
X = df.drop(columns=['Label']).copy()
print(f'  Distribuição do alvo: {dict(y.value_counts())}')
X.head()

## 3 · Mapa de discretização (faixas + fonte de cada corte)

**Coração metodológico.** Todos os cortes ficam num único dicionário, cada um com sua **fonte
oficial** — o padrão "faixa + citação" dos artigos do orientador (IMC→OMS, atividade física→
Guia BR 2021). O mapa é **comum aos dois desenhos** (IMC, idade, atividade física e os escores
existem em ambos), por isso o NB04 atende os dois sem alteração de regras.

`right=False` ⇒ intervalos `[a, b)` (use quando o limite superior pertence à próxima faixa,
como na OMS do IMC). `FAIXAS_QUANTIS` é só para escores compostos que **não** têm norma externa.

In [ ]:
# Faixas por DOMÍNIO (fonte oficial). Cortes no formato pd.cut.
FAIXAS = {
    'IMC': {                                   # Antropometria
        'bins':[0,18.5,25,30,100], 'labels':[0,1,2,3], 'right':False,
        'fonte':'OMS — classificação de IMC para adultos'},          # 0=Baixo 1=Normal 2=Sobrep 3=Obes
    'C008': {                                  # Idade — subdivisão do idoso (estudo é ≥60)
        'bins':[59,69,79,120], 'labels':[0,1,2], 'right':True,
        'fonte':'Subdivisão etária do idoso (risco crescente)'},     # 0=60–69 1=70–79 2=80+
    'P035': {                                  # Atividade física (dias/sem.)
        'bins':[-0.1,0,2,4,7], 'labels':[0,1,2,3], 'right':True,
        'fonte':'Guia de Atividade Física para a População Brasileira (2021)'},
    # TODO: acrescentar outras contínuas com norma (ex.: álcool→NIAAA) sempre com 'fonte'.
}

# Faixas por QUANTIS (equal-frequency) — só p/ escores SEM referência externa.
FAIXAS_QUANTIS = {
    'Escore_Inflamatorio': {'q':4, 'labels':[0,1,2,3], 'fonte':'Quartis da amostra (sem ref. externa)'},
    'Escore_Saudavel':     {'q':4, 'labels':[0,1,2,3], 'fonte':'Quartis da amostra (sem ref. externa)'},
}
print('  Faixas por domínio :', list(FAIXAS))
print('  Faixas por quantis :', list(FAIXAS_QUANTIS))

## 4 · Inventário das variáveis

Classifica as colunas em **discretizar** (numérica com faixa definida), **numérica não mapeada**
(decidir: criar faixa, mandar p/ quantis ou manter) e **já categórica** (dummies OHE 0/1 e
ordinais herdadas do NB03). Como os dois desenhos têm conjuntos de colunas diferentes (o Desenho 2
mantém as `Q*`), essa detecção automática é o que torna o NB04 robusto a ambos.

In [ ]:
mapeadas = set(FAIXAS) | set(FAIXAS_QUANTIS)
n_unicos = X.nunique(dropna=True)
col_numerica = [c for c in X.columns
                if pd.api.types.is_numeric_dtype(X[c]) and n_unicos[c] > 2]
a_discretizar  = [c for c in col_numerica if c in mapeadas]
nao_mapeadas   = [c for c in col_numerica if c not in mapeadas]
ja_categoricas = [c for c in X.columns if c not in col_numerica]

inv = pd.DataFrame({'coluna':X.columns,
    'n_unicos':[int(n_unicos[c]) for c in X.columns],
    'grupo':['discretizar' if c in a_discretizar
             else 'numérica_não_mapeada' if c in nao_mapeadas
             else 'já_categórica' for c in X.columns]})
print(inv['grupo'].value_counts(), '\n')
print('  A discretizar      :', a_discretizar)
print('  Numéricas s/ faixa :', nao_mapeadas, '  ← decidir o que fazer')
salvar_tab(DIR_TABELAS, inv.set_index('coluna'), 'etapa4_inventario_variaveis.csv')
inv

## 5 · Diagnóstico ANTES da discretização

O slide *Introdução11/12* mostra que o padrão só aparece **no histograma por intervalos**. Antes
de cortar, observamos cada contínua com os cortes marcados — o "será que os intervalos são os
mais adequados?" do slide.

In [ ]:
alvo_plot = [c for c in a_discretizar if c in FAIXAS and c in X.columns]  # só faixas de domínio têm cortes p/ desenhar
if alvo_plot:
    fig, axes = plt.subplots(1, len(alvo_plot), figsize=(5.2*len(alvo_plot), 4))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, alvo_plot):
        serie = pd.to_numeric(X[col], errors='coerce').dropna()
        ax.hist(serie, bins=30, color='#5B8DEF', edgecolor='white')
        for corte in FAIXAS[col]['bins'][1:-1]:
            ax.axvline(corte, color=COR_ARTRITE, ls='--', lw=1.2)
        ax.set_title(f'{col}\n(cortes: {FAIXAS[col]["bins"][1:-1]})'); ax.set_ylabel('Frequência')
    fig.suptitle('Etapa 5 — Distribuição ANTES (linhas = cortes de domínio)', y=1.04,
                 fontsize=13, weight='bold')
    plt.tight_layout(); salvar_fig(DIR_FIGURAS, 'etapa5_hist_antes.png'); plt.show()
else:
    print('  ⚠️ Nenhuma contínua mapeada presente.')

## 6 · Discretização não-supervisionada por domínio

Aplica `pd.cut` com os limites de `FAIXAS`. Cada nova coluna recebe sufixo `_cat`, é **ordinal**
(0=menor risco → k=maior) e mantém um **log** (variável, nº de faixas, fonte, distribuição) para
rastreabilidade.

In [ ]:
df_disc = X.copy()
log_disc = []
for col, spec in FAIXAS.items():
    if col not in df_disc.columns:
        print(f'  ⚠️ {col} ausente — pulado.'); continue
    nova = f'{col}_cat'
    df_disc[nova] = pd.cut(pd.to_numeric(df_disc[col], errors='coerce'),
                           bins=spec['bins'], labels=spec['labels'],
                           include_lowest=True, right=spec['right']).astype('float')
    dist = df_disc[nova].value_counts(dropna=False).sort_index().to_dict()
    log_disc.append({'variavel':col,'nova':nova,'metodo':'dominio',
                     'n_faixas':len(spec['labels']),'fonte':spec['fonte'],'distribuicao':dist})
    print(f'  ✅ {nova:24s} | {len(spec["labels"])} faixas | {spec["fonte"]}')
print('\n  Discretização por domínio concluída.')

## 7 · (Opcional) Discretização por frequência — `qcut`

O slide apresenta o **equal-frequency binning**. Use **apenas** para variáveis sem referência
externa — os **escores compostos** que você criou (não existe "OMS do Escore Inflamatório"). Com
norma médica, prefira sempre o domínio (Etapa 6).

In [ ]:
for col, spec in FAIXAS_QUANTIS.items():
    if col not in df_disc.columns:
        print(f'  ⚠️ {col} ausente — pulado.'); continue
    nova = f'{col}_cat'
    try:
        df_disc[nova] = pd.qcut(pd.to_numeric(df_disc[col], errors='coerce'),
                                q=spec['q'], labels=spec['labels'], duplicates='drop').astype('float')
        dist = df_disc[nova].value_counts(dropna=False).sort_index().to_dict()
        log_disc.append({'variavel':col,'nova':nova,'metodo':'quantis',
                         'n_faixas':spec['q'],'fonte':spec['fonte'],'distribuicao':dist})
        print(f'  ✅ {nova:24s} | {spec["q"]} quantis | {spec["fonte"]}')
    except ValueError as e:
        print(f'  ⚠️ {nova} não criada ({e})')
tab_disc = pd.DataFrame(log_disc)
salvar_tab(DIR_TABELAS, tab_disc.set_index('nova'), 'etapa6_7_log_discretizacao.csv')
tab_disc

## 8 · Verificação — prevalência da classe por faixa

O slide diz que **"reduzir a perda de informação é o objetivo da discretização"**. Uma faixa é
útil se Saudável × Com Artrite se distribuem de forma **diferente** entre os níveis. Este gráfico
empilhado é o mesmo das Figuras dos artigos de AVC/Hipertensão.

In [ ]:
cats = [r['nova'] for r in log_disc if r['nova'] in df_disc.columns]
if cats:
    fig, axes = plt.subplots(1, len(cats), figsize=(4.6*len(cats), 4))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, cats):
        tab = (pd.crosstab(df_disc[col], y, normalize='index')*100).rename(
               columns={0:'Saudável',1:'Com Artrite'})
        tab.plot(kind='bar', stacked=True, ax=ax,
                 color=[COR_SAUDAVEL, COR_ARTRITE], legend=(ax is axes[0]))
        ax.set_title(col); ax.set_xlabel('faixa'); ax.set_ylabel('% da faixa')
    fig.suptitle('Etapa 8 — Prevalência da classe por faixa (discrimina = faixa útil)',
                 y=1.04, fontsize=13, weight='bold')
    plt.tight_layout(); salvar_fig(DIR_FIGURAS, 'etapa8_prevalencia_por_faixa.png'); plt.show()
else:
    print('  ⚠️ Nenhuma faixa criada para avaliar.')

## 9 · Seleção de atributos — entropia + correlação

**Aqui (e só aqui) a entropia entra** — igual aos artigos (etapa "Remoção de Atributos por
Entropia/Correlação"). Entropia muito baixa ⇒ atributo quase constante (candidato a remoção);
correlação muito alta com `Label` ⇒ alerta de **variável-consequência** (slide
*Consequência×Causalidade*), diretamente ligado ao **data leakage do Desenho 2** (as `Q*`).

> Só **sinalizamos**; a remoção é decisão documentada (como os artigos: "não foi possível
> eliminar pois a entropia estava próxima entre os atributos").

In [ ]:
def entropia_shannon(serie):
    p = serie.value_counts(normalize=True, dropna=True); p = p[p > 0]
    return float(-(p*np.log2(p)).sum())

feat_final = cats + ja_categoricas
diag = []
for c in feat_final:
    if c not in df_disc.columns: continue
    col = pd.to_numeric(df_disc[c], errors='coerce')
    corr = np.corrcoef(col.fillna(col.median()), y)[0,1] if col.nunique() > 1 else 0.0
    diag.append({'feature':c, 'entropia':round(entropia_shannon(df_disc[c]),3),
                 'corr_abs_Label':round(abs(corr),3), 'n_unicos':int(df_disc[c].nunique())})

tab_sel = pd.DataFrame(diag).sort_values('entropia')
LIMIAR_ENTROPIA, LIMIAR_CORR = 0.10, 0.60
tab_sel['sinal'] = np.where(tab_sel['entropia'] < LIMIAR_ENTROPIA, 'baixa_entropia',
                   np.where(tab_sel['corr_abs_Label'] > LIMIAR_CORR, 'alta_corr_alvo', 'ok'))
salvar_tab(DIR_TABELAS, tab_sel.set_index('feature'), 'etapa9_selecao_entropia_correlacao.csv')
print(tab_sel['sinal'].value_counts(), '\n')
print('  ⚠️ Revisar (NÃO remover automaticamente):',
      tab_sel.loc[tab_sel.sinal != 'ok','feature'].tolist())
tab_sel.head(15)

## 10 · Codificação ordinal final e limpeza

Garante cada `_cat` como inteiro `0..k` e **remove as contínuas originais** já discretizadas
(evita "duplicações e redundâncias" do slide). Dummies OHE e ordinais herdadas do NB03 seguem
intactas.

In [ ]:
df_out = df_disc.copy()
originais = [c for c in (list(FAIXAS)+list(FAIXAS_QUANTIS))
             if f'{c}_cat' in df_out.columns and c in df_out.columns]
df_out.drop(columns=originais, inplace=True)
for c in [col for col in df_out.columns if col.endswith('_cat')]:
    df_out[c] = df_out[c].astype('Int64')
print(f'  Removidas (já discretizadas): {originais}')
print(f'  Dataset após discretização: {df_out.shape[0]:,} × {df_out.shape[1]}')
df_out.head()

## 11 · Exportação e relatório (desenho ativo)

Mesmo padrão do NB03: CSV pronto para o NB05 + JSON auditável (cada corte, fonte, método e os
sinais da seleção por entropia). Esta célula grava **o desenho ativo** (`DESENHO`); a Seção 12
grava os dois.

In [ ]:
df_final = df_out.copy(); df_final['Label'] = y.values
caminho_dataset = DIR_RESULTADOS + 'dataset_discretizado.csv'
df_final.to_csv(caminho_dataset, index=False, encoding='utf-8-sig')
print(f'  ✅ Dataset → {caminho_dataset}  ({df_final.shape[0]:,} × {df_final.shape[1]})')

relatorio = {
    'projeto':'Artrite e Reumatismo em Idosos Brasileiros — PNS 2019',
    'pesquisador':'Pedro Dias Soares', 'orientador':'Prof. Dr. Luis Enrique Zárate — PUC Minas',
    'notebook':'04_discretizacao', 'desenho':DESENHO, 'rotulo':cfg['rotulo'],
    'metodo':('Discretização não-supervisionada univariada por faixas de domínio (OMS/Guia BR); '
              'quantis só p/ escores sem referência externa. Entropia+correlação só p/ seleção.'),
    'parametros':{'RANDOM_STATE':RANDOM_STATE,'LIMIAR_ENTROPIA':LIMIAR_ENTROPIA,'LIMIAR_CORR':LIMIAR_CORR},
    'rastreabilidade':{
        'entrada':caminho_in, 'faixas_aplicadas':log_disc, 'continuas_removidas':originais,
        'selecao_atributos':tab_sel.to_dict('records'),
        'dataset_final':{'n_registros':int(df_final.shape[0]),'n_features':int(df_final.shape[1]-1),
            'distribuicao_label':{str(k):int(v) for k,v in y.value_counts().items()},
            'arquivo':caminho_dataset}}}
with open(DIR_RESULTADOS+'relatorio_discretizacao.json','w',encoding='utf-8') as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)
print(f'  ✅ Relatório → {DIR_RESULTADOS}relatorio_discretizacao.json')

## 12 · Processar **os dois desenhos** de uma vez

A Parte A acima percorreu **um** desenho com gráficos, para inspeção. Esta seção reúne a mesma
lógica numa função `discretizar_desenho(...)` e roda **ambos** (`puro` e `comorbidades`),
gravando os dois `dataset_discretizado.csv` e os dois `relatorio_discretizacao.json`.

A função reaproveita os **mesmos** `FAIXAS`/`FAIXAS_QUANTIS` (são clínicos, valem para os dois
desenhos) e detecta sozinha quais colunas existem em cada um — por isso atende tanto o 03 (sem
`Q*`) quanto o 03b (com `Q*`). É o que torna o NB04 "capaz de receber os resultados de ambos os
NB03".

In [ ]:
def discretizar_desenho(nome, cfg, FAIXAS, FAIXAS_QUANTIS,
                         LIMIAR_ENTROPIA=0.10, LIMIAR_CORR=0.60, plot=False):
    """Pipeline completo de discretização para UM desenho. Retorna (df_final, relatorio)."""
    figs, tabs = preparar_dirs(cfg['saida'])
    assert os.path.exists(cfg['entrada']), f"Entrada ausente: {cfg['entrada']} (rode o NB03/03b)."
    df_ = pd.read_csv(cfg['entrada'])
    y_  = df_['Label'].astype(int)
    Xd  = df_.drop(columns=['Label']).copy()

    # --- discretização por domínio + quantis ---
    log = []
    for col, spec in FAIXAS.items():
        if col not in Xd.columns: continue
        Xd[f'{col}_cat'] = pd.cut(pd.to_numeric(Xd[col], errors='coerce'),
            bins=spec['bins'], labels=spec['labels'], include_lowest=True,
            right=spec['right']).astype('float')
        log.append({'variavel':col,'nova':f'{col}_cat','metodo':'dominio',
                    'n_faixas':len(spec['labels']),'fonte':spec['fonte'],
                    'distribuicao':Xd[f'{col}_cat'].value_counts(dropna=False).sort_index().to_dict()})
    for col, spec in FAIXAS_QUANTIS.items():
        if col not in Xd.columns: continue
        try:
            Xd[f'{col}_cat'] = pd.qcut(pd.to_numeric(Xd[col], errors='coerce'),
                q=spec['q'], labels=spec['labels'], duplicates='drop').astype('float')
            log.append({'variavel':col,'nova':f'{col}_cat','metodo':'quantis',
                        'n_faixas':spec['q'],'fonte':spec['fonte'],
                        'distribuicao':Xd[f'{col}_cat'].value_counts(dropna=False).sort_index().to_dict()})
        except ValueError:
            pass

    # --- seleção entropia + correlação (apenas sinaliza) ---
    cats_ = [r['nova'] for r in log if r['nova'] in Xd.columns]
    ja_cat = [c for c in Xd.columns
              if not (pd.api.types.is_numeric_dtype(Xd[c]) and Xd[c].nunique(dropna=True) > 2)]
    diag = []
    for c in set(cats_ + ja_cat):
        col = pd.to_numeric(Xd[c], errors='coerce')
        ent = entropia_shannon(Xd[c])
        corr = np.corrcoef(col.fillna(col.median()), y_)[0,1] if col.nunique() > 1 else 0.0
        diag.append({'feature':c,'entropia':round(ent,3),'corr_abs_Label':round(abs(corr),3)})
    tab_sel_ = pd.DataFrame(diag).sort_values('entropia')
    tab_sel_['sinal'] = np.where(tab_sel_['entropia'] < LIMIAR_ENTROPIA, 'baixa_entropia',
                        np.where(tab_sel_['corr_abs_Label'] > LIMIAR_CORR, 'alta_corr_alvo', 'ok'))
    salvar_tab(tabs, tab_sel_.set_index('feature'), 'etapa9_selecao_entropia_correlacao.csv')

    # --- limpeza: remove contínuas já discretizadas, _cat como Int64 ---
    originais_ = [c for c in (list(FAIXAS)+list(FAIXAS_QUANTIS))
                  if f'{c}_cat' in Xd.columns and c in Xd.columns]
    Xd.drop(columns=originais_, inplace=True)
    for c in [col for col in Xd.columns if col.endswith('_cat')]:
        Xd[c] = Xd[c].astype('Int64')

    # --- exportação ---
    out = Xd.copy(); out['Label'] = y_.values
    csv_path = cfg['saida'] + 'dataset_discretizado.csv'
    out.to_csv(csv_path, index=False, encoding='utf-8-sig')
    rel = {
        'projeto':'Artrite e Reumatismo em Idosos Brasileiros — PNS 2019',
        'pesquisador':'Pedro Dias Soares','orientador':'Prof. Dr. Luis Enrique Zárate — PUC Minas',
        'notebook':'04_discretizacao','desenho':nome,'rotulo':cfg['rotulo'],
        'metodo':('Discretização não-supervisionada univariada por faixas de domínio (OMS/Guia BR); '
                  'quantis só p/ escores sem referência externa. Entropia+correlação só p/ seleção.'),
        'parametros':{'RANDOM_STATE':RANDOM_STATE,'LIMIAR_ENTROPIA':LIMIAR_ENTROPIA,'LIMIAR_CORR':LIMIAR_CORR},
        'rastreabilidade':{
            'entrada':cfg['entrada'],'faixas_aplicadas':log,'continuas_removidas':originais_,
            'selecao_atributos':tab_sel_.to_dict('records'),
            'dataset_final':{'n_registros':int(out.shape[0]),'n_features':int(out.shape[1]-1),
                'distribuicao_label':{str(k):int(v) for k,v in y_.value_counts().items()},
                'arquivo':csv_path}}}
    with open(cfg['saida']+'relatorio_discretizacao.json','w',encoding='utf-8') as f:
        json.dump(rel, f, ensure_ascii=False, indent=2)
    print(f'  ✅ [{nome}] {cfg["rotulo"]}: {out.shape[0]:,} × {out.shape[1]} → {csv_path}')
    return out, rel

# --- roda os DOIS desenhos ---
resultados = {}
for nome, c in CONFIG_DESENHOS.items():
    print(f'\n── Processando {nome} ──────────────────────────────')
    resultados[nome] = discretizar_desenho(nome, c, FAIXAS, FAIXAS_QUANTIS,
                                            LIMIAR_ENTROPIA, LIMIAR_CORR)
print('\n✅ Ambos os desenhos discretizados.')

## ✅ Sumário Final

In [ ]:
print('=' * 80)
print('  ✅  NOTEBOOK 04 — DISCRETIZAÇÃO CONCLUÍDA (ambos os desenhos)')
print('=' * 80)
print(f'  Faixas por domínio : {list(FAIXAS)}')
print(f'  Faixas por quantis : {list(FAIXAS_QUANTIS)}')
for nome, (out, rel) in resultados.items():
    info = rel['rastreabilidade']['dataset_final']
    print(f'  [{nome:13s}] {out.shape[0]:,} × {out.shape[1]} | alvo={info["distribuicao_label"]}'
          f' → {info["arquivo"]}')
print('  Próximo passo       : NB05 — Modelagem (Reg. Logística, Árvore, Random Forest; RUS+CV)')
print('  Sensibilidade (Desenho 2): Modelo A (com Q*) vs Modelo B (sem Q*) — ver leakage')
print('=' * 80)